# EXP 10 — one-click Kaggle run (preprocess, baseline, fine-tune, evaluate)

Import this notebook into Kaggle, set Accelerator = GPU and Internet = On, then use **Save Version → Save & Run All** so the ~1.5 h run continues on Kaggle's servers even if the browser is closed. Outputs: BLEU/chrF table and sample translations printed at the end.

In [ ]:
# Reliable one-click Kaggle pipeline
# Required Kaggle settings: Accelerator = GPU, Internet = On

import os
import subprocess
import torch

# Fail early instead of spending time downloading/training on CPU.
assert torch.cuda.is_available(), 'GPU not available. Enable a Kaggle GPU accelerator and restart the session.'
print('GPU:', torch.cuda.get_device_name(0))

# Install dependencies.
!pip -q install -U transformers datasets evaluate sentencepiece sacrebleu accelerate

# Clone only when the repository is not already present.
if not os.path.isdir('/kaggle/working/repo/.git'):
    subprocess.run([
        'git', 'clone',
        'https://github.com/Kartiken07/LOW-RESOURCE-NEURAL-MACHINE-TRANSLATION.git',
        '/kaggle/working/repo'
    ], check=True)

%cd /kaggle/working/repo

# 1. Preprocess Samanantar: 40k pairs, deterministic seed-42 split.
!python src/preprocess.py --hf-dataset samanantar --max-rows 40000 --out-dir data/processed

# 2. Evaluate pretrained baseline on 1,000 held-out test pairs.
!python src/evaluate.py --model facebook/nllb-200-distilled-600M --label "Pretrained (before fine-tuning)" --max-samples 1000

# 3. Fine-tune NLLB.
!python src/train.py --data-dir data/processed --output-dir nllb-en-as --epochs 2 --batch-size 4 --grad-accum 4 --lr 2e-5 --max-length 128 --seed 42

# 4. Evaluate the fine-tuned model on the same test subset.
!python src/evaluate.py --model nllb-en-as --label "Fine-tuned" --max-samples 1000

# 5. Display lightweight results.
import pandas as pd

print('\n=== METRICS ===')
display(pd.read_csv('results/metrics.csv'))

print('\n=== TRAINING TIME ===')
with open('nllb-en-as/training_time_minutes.txt') as f:
    print(f.read().strip(), 'minutes')

print('\n=== SAMPLE TRANSLATIONS ===')
display(pd.read_csv('results/sample_translations.csv'))

# Remove intermediate checkpoints while retaining the final model.
!rm -rf nllb-en-as/checkpoint-*
